[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/optical_rotation/optical_rotation.ipynb)

# Optical rotation, out and back

Quartz and a magnetic field both turn the polarization of light passing through them. Send the light back through the same medium, and they part ways: quartz turns it back, while the field keeps turning it, which is how an optical isolator stops reflections from returning to a laser.

The difference is orientation. Quartz turns light about its direction of travel, with a sense taken from that direction; the field turns light in a plane of its own, whichever way the light goes.

In the notation of polarization optics the quartz's turn reads as its specific rotation times the path length, which reverses with the direction of travel, and the field's as the Faraday rotation $V B L$, which does not.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, Orientation, stack
from numga.algebras import STA
from examples.animation import save_figure
from examples.orientation.optical_rotation import render

In [ ]:
# Spacetime, with space positive, `x+y+z+t-`.
ga = STA
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
# Directions, of travel and of the electric field across the path, are antivectors.
Direction = ga.gatype.antivector()            # Antivector-
Polarization = ga.gatype.antivector()         # Antivector-
Mirror = ga.gatype.vector()                   # Vector+
Field = ga.gatype.bivector()                  # Bivector+
TurnRate = ga.gatype.bivector()               # Bivector+
# A turn per distance with a sense of its own: measured against a handedness.
RotaryPower = ga.gatype(ga.subspace.scalar(), orientation=Orientation.intrinsic)   # Scalar-
# The direction of time of an observer at rest with the media, which pairs to one with `mv.t`; the dual
# carries the minus of t's square.
time = -mv.t.dual()                           # Antivector-

Light travels along `direction`, and its polarization is the direction of its electric field, across the path; both are antivectors, intrinsic like all geometry. An optically active medium turns the polarization about the direction of travel by its rotary power per distance. The light sweeps a plane through spacetime, its direction joined with time, `direction & time`: an intrinsic bivector oriented by the travel, which as a turn lies across the path. An intrinsic bivector is not a rate, and `exp` refuses it. The rotary power is what makes it one: the medium's turn has a sense of its own, measured against a handedness, so the rotary power is an intrinsic scalar, and the product of the two is an extrinsic bivector that exponentiates.

A magnetic field turns the polarization in the part of its bivector that lies in that plane, `sweep * field.dot(sweep)`, by the Verdet constant per distance and field. The swept plane enters twice, so reversing the light leaves that part unchanged.

In [ ]:
length = 2.0                                          # mm through the medium
n_samples = 9                                         # polarization samples along each leg
depths = np.linspace(0, length, n_samples)
verdet = np.deg2rad(22.5)                             # rad per mm per tesla, illustrative

direction: Direction = mv.z.dual()
start: Polarization = mv.x.dual()
# Quartz turns 21.7 degrees per mm; measured against the lab's handedness, the basis volume x ^ y ^ z.
rotary_power: RotaryPower = mv(RotaryPower, [np.deg2rad(22.5)])   # Scalar-
field: Field = mv.xy * 1.0                            # Bivector+, one tesla along z


def out_and_back(out_rate: TurnRate, back_rate: TurnRate, start: Polarization) -> Polarization:
    # Out to a reflector at the far end and back, [legs, depths]: the light leaves where it entered.
    out = (out_rate * (-depths / 2)).exp() >> start
    back = (back_rate * (-depths / 2)).exp() >> out[-1]
    return stack([out, back])


def through_media(rotary_power: RotaryPower, field: Field, start: Polarization) -> Polarization:
    # [media, legs, depths]: the optically active medium, then the magnetic field.
    # The plane the light sweeps through time, out and back.
    sweep, back_sweep = direction & time, -direction & time                         # Bivector-
    active = out_and_back(sweep * rotary_power, back_sweep * rotary_power, start)
    faraday = out_and_back(sweep * field.dot(sweep) * verdet, back_sweep * field.dot(back_sweep) * verdet, start)
    return stack([active, faraday])


paths: Polarization = through_media(rotary_power, field, start)    # [media, legs, depths] Antivector-
# Malus's law: the share of the returning light that passes the polarizer it entered through.
passed = paths[:, 1, -1].dot(start).squared()                      # [media] Scalar+

In [ ]:
print("Share of the returning light that passes the polarizer it entered through")
for medium, share in zip(("Quartz", "Field"), passed):
    print(medium)
    print(share.coefficient_table())

The mirror image of the setup reflects the medium, the field and the incoming light alike. The rotary power, an intrinsic scalar, changes sign: the mirror image of right-handed quartz is left-handed quartz. The field, a bivector, reflects as a plane does. Each mirrored setup then gives the mirror image of the original light, at every depth on both legs.

The figure unfolds each path along the beam, out to the reflector and back, with quartz on the left and the field on the right, the original setups above their mirror images. Each line is the polarization across the beam, orange on the way out and teal on the way back; the dashed lines at both ends are the polarization the light entered with.

In [ ]:
mirror: Mirror = mv.x                                 # the plane x = 0, holding the path
mirrored: Polarization = through_media(mirror >> rotary_power, mirror >> field, mirror >> start)   # [media, legs, depths] Antivector-
mismatch = (mirrored - (mirror >> paths)).squared()   # [media, legs, depths] Scalar+

polarizations = stack([paths, mirrored], axis=1)      # [media, setups, legs, depths] Antivector-
save_figure(render.draw_isolator(polarizations, depths), "optical_rotation")

In [ ]:
print("Largest squared difference between the mirrored setups and the mirror image of the originals")
print(mismatch.to_array().max())

In [ ]:
# checks
# The quartz returns the light as it entered; the field returns it crossed, so the polarizer stops it. The
# mirrored setups give the mirror image of the original light, at every depth on both legs.
active, faraday = paths[:, 1, -1]
assert (active - start).squared().to_array() < 1e-24
assert abs(faraday.dot(start).to_array()) < 1e-12
assert mismatch.to_array().max() < 1e-24